# TwelveData - Khám phá thư viện TwelveData

Notebook này giúp bạn khám phá nhanh thư viện `twelve_data`:
- Kiểm tra kết nối API qua biến môi trường
- Kiểm tra dữ liệu real-time qua websocket
- Lấy dữ liệu nến từ quá khứ
- Lấy tin tức chung của sàn chứng khoán

## 1. Cấu hình

In [1]:
import sys
import os
import asyncio
from datetime import datetime
from IPython.display import clear_output, display

# Thêm thư mục gốc vào path để import src
sys.path.append(os.path.abspath('..'))

from src.config import Config
from src.streamer import twelvedata_streamer

In [2]:
# Kiểm tra key
if not Config.TWELVEDATA_API_KEY:
    print("Error: TWELVEDATA_API_KEY is not set in the environment variables.")
    sys.exit(1)
else:
    print("TWELVEDATA_API_KEY: ", Config.TWELVEDATA_API_KEY[0:4] + "...")  # In một phần của key để xác nhận

TWELVEDATA_API_KEY:  2ac6...


## 2. Giao dịch - Cập nhật giá cuối cùng

Finhub có hỗ trợ cho bản free Websocket để lấy giá liên tục từ chứng khoán thị trường Hoa Kỳ

In [ ]:
# 1. Đổi lại danh sách symbols theo chuẩn của Twelve Data
symbols_to_track = ["AAPL", "BTC/USD", "EUR/USD"]
api_key = Config.TWELVEDATA_API_KEY 

# Khởi tạo dữ liệu trống
latest_prices = {symbol: {"price": "Đang đợi...", "volume": "0", "time": "--:--:--"} for symbol in symbols_to_track}

async def run_tracker():
    print("🚀 Đang kết nối server Twelvedata...")
    
    try:
        # Nhận từng gói dữ liệu một từ streamer
        async for data in twelvedata_streamer(api_key, symbols_to_track):
            
            # 2. Lấy dữ liệu bằng các key chuẩn của Twelve Data
            symbol = data.get('symbol')
            if not symbol or symbol not in latest_prices:
                continue

            # 3. Định dạng lại thời gian (Timestamp của Twelve Data tính bằng Giây)
            timestamp = data.get('timestamp')
            if timestamp:
                dt_object = datetime.fromtimestamp(timestamp) # KHÔNG cần chia 1000.0 nữa
                time_str = dt_object.strftime('%H:%M:%S')
            else:
                time_str = "--:--:--"
            
            # Lấy giá và volume (Twelve data thường dùng 'day_volume')
            price = data.get('price', 0)
            volume = data.get('day_volume', 0)
            
            # Cập nhật thông tin vào dictionary
            latest_prices[symbol] = {
                "price": f"{price:,.2f}",
                # Volume của crypto/forex có thể là số thập phân, cổ phiếu là số nguyên
                "volume": f"{volume:,.2f}" if isinstance(volume, float) else f"{volume}", 
                "time": time_str
            }
            
            # Xóa cell và in lại bảng mới
            clear_output(wait=True)
            print(f"📊 BẢNG GIÁ REAL-TIME ({datetime.now().strftime('%d/%m/%Y')})")
            print("-" * 65)
            print(f"{'Mã':<18} | {'Giá (USD)':>12} | {'KL trong ngày':>13} | {'Thời gian'}")
            print("-" * 65)
            
            for sym, info in latest_prices.items():
                print(f"{sym:<18} | {info['price']:>12} | {info['volume']:>13} | {info['time']}")
            
            print("-" * 65)
            print("💡 Mẹo: Nhấn nút Stop (Interrupt) để dừng cập nhật.")

    except asyncio.CancelledError:
        print("\n⏹️ Đã dừng Task thành công.")
    except Exception as e:
        print(f"\n❌ Có lỗi xảy ra: {e}")

# Chạy task
stream_task = asyncio.create_task(run_tracker())

📊 BẢNG GIÁ REAL-TIME (17/04/2026)
-----------------------------------------------------------------
Mã                 |    Giá (USD) | KL trong ngày | Thời gian
-----------------------------------------------------------------
AAPL               |  Đang đợi... |             0 | --:--:--
BTC/USD            |    74,850.26 |             0 | 11:41:00
EUR/USD            |         1.18 |             0 | 11:41:00
-----------------------------------------------------------------
💡 Mẹo: Nhấn nút Stop (Interrupt) để dừng cập nhật.

⏹️ Đã dừng Task thành công.


Nếu muốn dừng, bạn chạy lệnh dưới đây

In [4]:
stream_task.cancel() # Dừng task nếu cần thiết

True

Do twelveData chỉ hỗ trợ cho 8 symbols nên sẽ không được sử dụng nhiều